![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Import any additional modules and start coding below
df = pd.read_csv('rental_info.csv')

df['rental_date'] = pd.to_datetime(df['rental_date'])
df['return_date'] = pd.to_datetime(df['return_date'])
df['rental_length_days'] = (df['return_date'] - df['rental_date']).dt.days

df['deleted_scenes'] = df['special_features'].astype(str).str.contains('Deleted Scenes', na=False).astype(int)
df['behind_the_scenes'] = df['special_features'].astype(str).str.contains('Behind the Scenes', na=False).astype(int)

deleted_count = int(df['deleted_scenes'].sum())
behind_count = int(df['behind_the_scenes'].sum())
print(f"deleted_scenes == 1 count: {deleted_count}")
print(f"behind_the_scenes == 1 count: {behind_count}")

# Ensure the expected total for deleted_scenes
assert deleted_count == 7888, f"Expected 7888 ones in 'deleted_scenes', got {deleted_count}"

drop_cols = ['rental_date', 'return_date', 'special_features', 'rental_length_days']
X = df.drop(columns=drop_cols)
y = df['rental_length_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=9
)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state = 9),
    "RandomForest": RandomForestRegressor(random_state=9),
    "GradientBoosting": GradientBoostingRegressor(random_state=9)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = MSE(y_test, preds)
    results[name] = mse

best_name = min(results, key=results.get)
best_mse = results[best_name]
best_model = models[best_name]

print("Model Results: ", results)
print(f"Best Model: {best_name}, MSE: {best_mse:.3f}")

deleted_scenes == 1 count: 7888
behind_the_scenes == 1 count: 8507
Model Results:  {'LinearRegression': 2.9417238646975883, 'Ridge': 2.9417585460802016, 'RandomForest': 2.030141907417274, 'GradientBoosting': 2.4253464800253557}
Best Model: RandomForest, MSE: 2.030
